# 01 - The data, and what a softmax cannot tell you

Two things are established here, both without training anything:

1. **The dataset is a real benchmark, not a toy.** The procedural generator
   produces dermoscopy-like images containing the failure modes that matter for
   uncertainty research - soft boundaries, low contrast, hair and ruler
   occluders, non-convex outlines - and it needs no download.
2. **The evidential decomposition separates two situations a confidence score
   reports identically.** This is the premise the whole method rests on, so it
   is checked numerically before any model is fitted.

In [ ]:
# Run from the repository root, or from notebooks/ - both work.
import sys, os
from pathlib import Path

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
    os.chdir(root)
sys.path.insert(0, str(root / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

torch.set_num_threads(max(1, (os.cpu_count() or 2)))
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
print("repo root :", root)
print("torch     :", torch.__version__, "| cuda:", torch.cuda.is_available())

## The generator

Every sample is a pure function of one integer, so a dataset is reproducible
from a seed. Each carries the latent factors behind it, which lets difficulty be
correlated with error later on.

In [ ]:
from evissl.data import generate_dataset, generate_sample
from evissl.viz import plot_samples, use_style

use_style()
image, mask, params = generate_sample(7, 128)
print(f"image {image.shape} {image.dtype}   mask {mask.shape} values {np.unique(mask)}")
print(f"contrast       {params.contrast:.3f}")
print(f"edge softness  {params.edge_softness:.2f} px")
print(f"lesion area    {params.area_fraction:.3f} of frame")
print(f"hair strands   {params.n_hairs}")
print(f"difficulty     {params.difficulty():.3f}")

# Determinism, asserted rather than assumed.
again, _, _ = generate_sample(7, 128)
assert np.array_equal(image, again)
print("\nsame seed -> identical image: True")

In [ ]:
images, masks, params = generate_dataset(12, 128, seed=1)
difficulty = np.array([p.difficulty() for p in params])
fig = plot_samples(images, masks, difficulty, n=8,
                   title="Synthetic dermoscopy: contour in green")
plt.show()

### Difficulty is broadly distributed

A benchmark where every sample is easy cannot discriminate between methods. The
generator draws a configurable fraction of samples from a hard regime (low
contrast, soft edge, heavy occlusion).

In [ ]:
images, masks, params = generate_dataset(300, 96, seed=2)
frame = pd.DataFrame({
    "contrast": [p.contrast for p in params],
    "edge_softness": [p.edge_softness for p in params],
    "area_fraction": [p.area_fraction for p in params],
    "n_hairs": [p.n_hairs for p in params],
    "difficulty": [p.difficulty() for p in params],
})
display(frame.describe().round(3))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, column in zip(axes, ["contrast", "edge_softness", "difficulty"]):
    ax.hist(frame[column], bins=30, color="#009E73", alpha=0.85)
    ax.set_xlabel(column.replace("_", " "))
    ax.set_ylabel("count")
axes[0].set_title("Pigment contrast against skin")
axes[1].set_title("Boundary blur (px)")
axes[2].set_title("Composite difficulty")
plt.tight_layout(); plt.show()

print(f"lesion pixel fraction: mean {masks.mean():.4f}")
print(f"class imbalance      : roughly 1 lesion pixel per "
      f"{(1 - masks.mean()) / masks.mean():.1f} background pixels")

## The premise: vacuity is not dissonance

The network emits non-negative evidence `e = softplus(logits)`, giving a
Dirichlet with `alpha = e + 1`. Subjective logic splits the resulting opinion
into belief per class, **vacuity** (`K/S`, epistemic - not enough evidence) and
**dissonance** (aleatoric - the evidence conflicts).

The table below is the argument for the whole method. Rows 1 and 2 have the same
predicted probability and opposite epistemic states.

In [ ]:
from evissl.uncertainty import belief, dirichlet_alpha, evidential_output, vacuity

cases = {
    "no evidence            [-60, -60]": torch.full((1, 2, 4, 4), -60.0),
    "conflicting evidence   [+30, +30]": torch.full((1, 2, 4, 4), 30.0),
    "confident lesion       [-30, +30]": torch.tensor([[[[-30.0]], [[30.0]]]]),
    "weak lesion evidence   [ -1,  +1]": torch.tensor([[[[-1.0]], [[1.0]]]]),
}
rows = []
for name, logits in cases.items():
    out = evidential_output(logits)
    rows.append({
        "case": name,
        "P(lesion)": float(out.lesion_prob.mean()),
        "vacuity (epistemic)": float(out.vacuity.mean()),
        "dissonance (aleatoric)": float(out.dissonance.mean()),
        "strength S": float(out.strength.mean()),
    })
display(pd.DataFrame(rows).round(4).set_index("case"))

print("Rows 1 and 2 share P(lesion) = 0.5 exactly.")
print("A softmax reports one number and cannot separate them.")
print("A confidence threshold therefore discards both - including row 2,")
print("which is a real, informative boundary observation.")

In [ ]:
# The subjective-logic identity, checked on random evidence.
alpha = dirichlet_alpha(torch.randn(4, 2, 32, 32) * 5)
total = belief(alpha).sum(dim=1) + vacuity(alpha)
print(f"max |sum_k b_k + u - 1| = {float((total - 1).abs().max()):.2e}")

### The decomposition over the whole plane

Sweeping both logits shows the structure: vacuity is high only where *both*
evidences are small, while dissonance is high where both are large and similar.
The diagonal is where they diverge most - and it is exactly the set of pixels a
confidence threshold treats as one thing.

In [ ]:
grid = torch.linspace(-6, 8, 160)
e0, e1 = torch.meshgrid(grid, grid, indexing="ij")
logits = torch.stack([e0, e1]).unsqueeze(0)
out = evidential_output(logits)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
panels = [
    (out.prob[0, 1], "P(lesion)", "viridis"),
    (out.vacuity[0], "vacuity  (epistemic)", "magma"),
    (out.dissonance[0], "dissonance  (aleatoric)", "magma"),
]
for ax, (data, title, cmap) in zip(axes, panels):
    im = ax.imshow(data.numpy(), origin="lower", cmap=cmap, vmin=0, vmax=1,
                   extent=[-6, 8, -6, 8], aspect="auto")
    ax.set_xlabel("logit for background"); ax.set_ylabel("logit for lesion")
    ax.set_title(title)
    ax.plot([-6, 8], [-6, 8], color="white", lw=1.0, ls="--", alpha=0.7)
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle("Along the dashed diagonal P = 0.5 everywhere, "
             "yet vacuity falls from 1 to 0 and dissonance rises from 0 to 1", y=1.04)
plt.tight_layout(); plt.show()

## What each weighting rule would actually keep

With a teacher that has moderate, near-evenly-split evidence, FixMatch's
threshold retains nothing while the evidential rule retains a share
proportional to the belief mass present. This is the mechanism, isolated from
any training run.

In [ ]:
from evissl.config import SemiConfig
from evissl.losses import consistency_loss

student = torch.randn(4, 2, 32, 32)
rows = []
for label, magnitude in [("very weak", -3.0), ("weak", 0.0), ("moderate", 2.0), ("strong", 6.0)]:
    teacher = torch.full((4, 2, 32, 32), magnitude)
    teacher[:, 1] += 0.2                        # a slight, genuine lean towards lesion
    entry = {"teacher evidence": label}
    for method in ("mean_teacher", "fixmatch", "evidential"):
        out = consistency_loss(student, teacher, SemiConfig(method=method))
        entry[method] = out.mask_rate
    rows.append(entry)

table = pd.DataFrame(rows).set_index("teacher evidence")
display(table.round(4))

ax = table.plot(kind="bar", figsize=(7.5, 3.8),
                color=["#0072B2", "#E69F00", "#009E73"], rot=0)
ax.set_ylabel("effective fraction of pixels used")
ax.set_title("How much of the unlabelled signal each rule keeps")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

print("FixMatch is all-or-nothing and is at zero until the teacher is already sure.")
print("Mean Teacher uses everything, including pixels the teacher knows nothing about.")
print("The evidential rule scales continuously with the evidence actually present.")

Next: **02_train_and_compare.ipynb** trains all four rules through one shared
loop and tests whether that difference shows up in the metrics.